# Gated Attention for LLMs — Version 3: + PNG Gate
Extends v2 (G1–G5 ablation) with **PNG (Patch-Norm Gating)** — a norm-aware gate from the ViT literature adapted to GPT-2 attention.

| | |
|---|---|
| **Model** | GPT-2 small (124M params, 12 layers, 12 heads) |
| **Dataset** | WikiText-103 |
| **Backbone** | **Unfrozen** — full fine-tuning |
| **New gate** | PNG: `G = σ(X·Wθ − β·\|\|X\|\|₂·e)` |
| **Version** | v3 — loads G1–G5 + Baseline from v2, trains only PNG |

**PNG gate equation:**
```
x_norm = ||x||₂  (B, N, 1)
G = sigmoid(x @ W_θ  −  beta * x_norm * e)   (B, N, H)
output = SDPA_out * G   (applied post-SDPA, same position as G1)
```
`W_θ ∈ ℝ^{d×H}`, `e ∈ ℝ^H` (learnable per-head scale), `beta=0.1` (fixed hyperparameter).

## 0. Setup

In [ ]:
import os, types, math, json, shutil, glob as _glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy.stats import pearsonr
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from datasets import load_dataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'



TEST =False

SEQ_LEN      = 512
BATCH_SIZE   = 4    if TEST else 8
EPOCHS       = 1    if TEST else 5
LR           = 2e-5
GRAD_CLIP    = 1.0
TRAIN_TOKENS = 25_600  if TEST else 2_000_000
VAL_TOKENS   = 10_240  if TEST else None
TEST_TOKENS  = 10_240  if TEST else None
EVAL_BATCHES = 5       if TEST else None
SINK_BATCHES = 3       if TEST else 10
GATE_BATCHES = 5       if TEST else 20
DISPLAY_LEN  = 20      if TEST else 40
EVAL_EVERY   = 5       if TEST else 50

# v2 checkpoints dir (read-only source)
V2_CKPT_DIR  = Path('checkpoints_llm_unfrozen')
# v3 checkpoints dir (PNG lives here; v2 ckpts are copied in)
CKPT_DIR     = Path('checkpoints_llm_v3' + ('_test' if TEST else ''))
CKPT_DIR.mkdir(exist_ok=True)

def _safe_name(n):
    return n.replace(' ', '_').replace('—', '').replace('(', '').replace(')', '').replace('/', '')

print(f'torch {torch.__version__} | device {DEVICE}')
print(f'TEST={TEST}  epochs={EPOCHS}  batch={BATCH_SIZE}  lr={LR}')
print(f'v2 ckpt dir : {V2_CKPT_DIR.resolve()}')
print(f'v3 ckpt dir : {CKPT_DIR.resolve()}')

## 0b. Restore v2 Checkpoints from Kaggle Input

In [ ]:
# ── Copy v2 checkpoints (G1–G5 + Baseline) into CKPT_DIR ─────────────────
# Kaggle path pattern:
#   /kaggle/input/notebooks/<user>/gated-llm/checkpoints_llm_unfrozen/
#
# Attach your v2 notebook output as an input dataset before running.

_src_dirs = (
    _glob.glob('/kaggle/input/**/checkpoints_llm_unfrozen', recursive=True) +
    _glob.glob('/kaggle/input/notebooks/**/checkpoints_llm_unfrozen', recursive=True)
)
_src_dirs = sorted(set(_src_dirs))

# Also check if v2 dir exists locally (same Kaggle session or local run)
if V2_CKPT_DIR.exists():
    _src_dirs = [str(V2_CKPT_DIR)] + _src_dirs

if _src_dirs:
    _src = Path(_src_dirs[0])   # prefer local v2 dir first
    print(f'v2 source: {_src}')
    _copied = 0
    for _f in _src.iterdir():
        if not _f.name.endswith('.pth'): continue
        _dst = CKPT_DIR / _f.name
        if not _dst.exists():
            shutil.copy2(_f, _dst)
            print(f'  Copied : {_f.name}  ({_f.stat().st_size/1e6:.1f} MB)')
            _copied += 1
        else:
            print(f'  Exists : {_f.name}')
    print(f'Done — {_copied} file(s) copied.')
else:
    print('WARNING: no v2 checkpoint source found.')
    print('G1–G5 + Baseline will not be available for analysis.')

print('\nCheckpoints in v3 dir:')
for _f in sorted(CKPT_DIR.iterdir()):
    if _f.suffix == '.pth':
        _d = torch.load(_f, map_location='cpu', weights_only=False)
        _ed = _d.get('epochs_done', '?') if isinstance(_d, dict) else '?'
        print(f'  {_f.name:<45} epochs_done={_ed}')

## 1. Dataset — WikiText-103

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

raw = load_dataset('wikitext', 'wikitext-103-raw-v1')

def tokenize_and_chunk(split, max_tokens=None):
    ids_list, collected = [], 0
    limit = max_tokens or int(1e18)
    for row in raw[split]:
        text = row['text'].strip()
        if not text: continue
        toks = tokenizer(text, add_special_tokens=False, return_tensors='pt')['input_ids'][0]
        if collected + len(toks) > limit:
            toks = toks[:limit - collected]
        ids_list.append(toks)
        collected += len(toks)
        if collected >= limit: break
    all_ids = torch.cat(ids_list)
    n = len(all_ids) // SEQ_LEN
    return all_ids[:n * SEQ_LEN].reshape(n, SEQ_LEN)

print('Tokenising ...')
train_ids = tokenize_and_chunk('train',      max_tokens=TRAIN_TOKENS)
val_ids   = tokenize_and_chunk('validation', max_tokens=VAL_TOKENS)
test_ids  = tokenize_and_chunk('test',       max_tokens=TEST_TOKENS)

class TokenDataset(Dataset):
    def __init__(self, c): self.c = c
    def __len__(self):     return len(self.c)
    def __getitem__(self, i): return self.c[i]

train_loader = DataLoader(TokenDataset(train_ids), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(TokenDataset(val_ids),   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(TokenDataset(test_ids),  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train {len(train_ids):>5} chunks | Val {len(val_ids):>4} | Test {len(test_ids):>4}')
print(f'Steps/epoch: {len(train_loader)}')

## 2. Gate Modules — G1–G5 + PNG

| Gate | Position | Equation |
|------|----------|----------|
| G1 | Post-SDPA | `out *= σ(X·Wθ)` |
| G2 | Value proj | `v *= σ(X·Wθ)` |
| G3 | Key proj | `k *= σ(X·Wθ)` |
| G4 | Query proj | `q *= σ(X·Wθ)` |
| G5 | Post-concat | `out *= σ(X·Wθ)` |
| **PNG** | **Post-SDPA (norm-aware)** | **`out *= σ(X·Wθ − β·‖X‖₂·e)`** |

PNG differs from G1 by subtracting a norm-scaled bias: tokens with large norm get **extra suppression**.

In [ ]:
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention

# ── Shared helpers ─────────────────────────────────────────────────────────
def _split_heads(t, nh, hd):
    return t.view(t.size()[:-1] + (nh, hd)).permute(0, 2, 1, 3)

def _merge_heads(t, nh, hd):
    return t.permute(0, 2, 1, 3).contiguous().view(t.size(0), t.size(2), nh * hd)

def _apply_gate(t_bhnd, g_bnh):
    return t_bhnd * g_bnh.permute(0, 2, 1).unsqueeze(-1)


# ── HeadGate: simple sigmoid gate (G1–G5) ─────────────────────────────────
class HeadGate(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.W = nn.Parameter(torch.zeros(d_model, n_heads))

    def forward(self, x):
        return torch.sigmoid(x @ self.W)   # (B, N, H)


# ── PNGGate: norm-aware sigmoid gate ──────────────────────────────────────
class PNGGate(nn.Module):
    """
    PNG gate (adapted from arXiv 2505.06708 / ViT PNG paper).
    G = sigmoid(X @ W_theta  -  beta * ||X||_2 * e)
    W_theta: (d_model, n_heads)  — directional gating
    e:       (n_heads,)          — per-head norm sensitivity scale
    beta:    scalar hyperparameter (default 0.1)
    """
    def __init__(self, d_model, n_heads, beta=0.1):
        super().__init__()
        self.W    = nn.Parameter(torch.zeros(d_model, n_heads))
        self.e    = nn.Parameter(torch.zeros(n_heads))
        self.beta = beta

    def forward(self, x):
        # x: (B, N, d_model)
        linear = x @ self.W                          # (B, N, H)
        x_norm = x.norm(dim=-1, keepdim=True)        # (B, N, 1)
        norm_bias = self.beta * x_norm * self.e      # (B, N, H)
        return torch.sigmoid(linear - norm_bias)     # (B, N, H)


# ── Patched forward (shared by all gate types) ─────────────────────────────
def make_gated_forward(attn_module, gate, pos):
    n_head    = attn_module.num_heads
    head_size = attn_module.head_dim
    embed_dim = attn_module.embed_dim

    def forward(self, hidden_states, **kwargs):
        B, N, _ = hidden_states.shape
        x = hidden_states
        g = gate(x)   # (B, N, H)

        layer_past        = kwargs.get('past_key_values', kwargs.get('layer_past', None))
        attention_mask    = kwargs.get('attention_mask', None)
        use_cache         = kwargs.get('use_cache', False)
        output_attentions = kwargs.get('output_attentions', False)

        qkv = self.c_attn(hidden_states)
        q, k, v = qkv.split(self.embed_dim, dim=2)
        q = _split_heads(q, n_head, head_size)
        k = _split_heads(k, n_head, head_size)
        v = _split_heads(v, n_head, head_size)

        if pos == 'G4': q = _apply_gate(q, g)
        if pos == 'G3': k = _apply_gate(k, g)
        if pos == 'G2': v = _apply_gate(v, g)

        if layer_past is not None:
            if isinstance(layer_past, tuple):
                k = torch.cat([layer_past[0], k], dim=-2)
                v = torch.cat([layer_past[1], v], dim=-2)
        present = (k, v) if use_cache else None

        if output_attentions or getattr(self, '_capture_attn', False):
            scale  = head_size ** -0.5
            scores = (q * scale) @ k.transpose(-2, -1)
            if attention_mask is not None:
                scores = scores + attention_mask
            if layer_past is None:
                Nq, Nk = q.size(-2), k.size(-2)
                cm = torch.triu(torch.ones(Nq, Nk, device=x.device, dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None, None], float('-inf'))
            attn_w   = scores.softmax(-1)
            attn_w   = self.attn_dropout(attn_w)
            attn_out = attn_w @ v
            if getattr(self, '_capture_attn', False):
                self._last_attn = attn_w.detach()
        else:
            attn_out = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attention_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=(layer_past is None),
            )
            attn_w = None

        # G1 / PNG both applied post-SDPA
        if pos in ('G1', 'PNG'):
            attn_out = _apply_gate(attn_out, g)
        # ── Capture gate & norm for ALL positions (Fig 5/6/7) ──
        if getattr(self, '_capture_gate', False):
            self._last_gate   = g.detach()
            self._last_x_norm = x.norm(dim=-1).detach()

        attn_out = _merge_heads(attn_out, n_head, head_size)

        if pos == 'G5':
            attn_out = (
                attn_out.view(B, N, n_head, head_size) * g.unsqueeze(-1)
            ).view(B, N, embed_dim)

        attn_out = self.c_proj(attn_out)
        attn_out = self.resid_dropout(attn_out)

        outputs = (attn_out, present)
        if output_attentions:
            outputs += (attn_w,)
        return outputs

    return types.MethodType(forward, attn_module)


print('Gate modules ready: HeadGate (G1–G5) + PNGGate (PNG).')

## 3. Model Builder

In [ ]:
def inject_gates_gpt2(model, pos):
    d_model = model.config.n_embd
    n_heads = model.config.n_head
    gate_list = []
    for block in model.transformer.h:
        if pos == 'PNG':
            g = PNGGate(d_model, n_heads, beta=0.1)
        else:
            g = HeadGate(d_model, n_heads)
        block.attn.forward = make_gated_forward(block.attn, g, pos)
        gate_list.append(g)
    model.gate_params = nn.ModuleList(gate_list)
    return model


def build_model(pos='baseline'):
    m = GPT2LMHeadModel.from_pretrained('gpt2')
    if pos != 'baseline':
        inject_gates_gpt2(m, pos)
    for p in m.parameters():
        p.requires_grad_(True)
    m = m.to(DEVICE)
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in m.parameters())
    gate_extra = sum(p.numel() for p in m.gate_params.parameters()) if hasattr(m, 'gate_params') else 0
    print(f'  [{pos}]  trainable={n_train:,} / {n_total:,}'
          + (f'  (+{gate_extra:,} gate params)' if gate_extra else ''))
    return m

print('build_model() ready (supports PNG).')

## 4. Evaluation — Perplexity

In [ ]:
@torch.no_grad()
def evaluate_ppl(model, loader, max_batches=EVAL_BATCHES):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches: break
        ids = batch.to(DEVICE)
        B, N = ids.shape
        out = model(ids, labels=ids)
        total_loss   += out.loss.item() * B * (N - 1)
        total_tokens += B * (N - 1)
    return math.exp(total_loss / total_tokens)

print('evaluate_ppl ready.')

## 5. Training Loop

In [ ]:
RESULTS_FILE = CKPT_DIR / 'results.json'


def train_one(model, name='model', epochs=EPOCHS, lr=LR, ckpt_path=None, resume_ckpt=None):
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=lr, weight_decay=0.01)
    total_steps = epochs * len(train_loader)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(total_steps, 1))

    log = {'step': [], 'train_loss': [], 'val_ppl': []}
    start_epoch, step = 0, 0

    if resume_ckpt and Path(resume_ckpt).exists():
        ckpt = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['state'])
        opt.load_state_dict(ckpt['optimizer'])
        sched.load_state_dict(ckpt['scheduler'])
        log, start_epoch = ckpt['log'], ckpt['epoch']
        step = start_epoch * len(train_loader)
        print(f'  Resumed from {Path(resume_ckpt).name} (epoch {start_epoch})')
    elif ckpt_path and Path(ckpt_path).exists():
        data = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(data['state'])
        log         = data.get('log', log)
        start_epoch = data.get('epochs_done', 0)
        step        = start_epoch * len(train_loader)
        if start_epoch >= epochs:
            print(f'  Already {start_epoch} epochs done — skipping.')
            return log

    print(f'  Training epochs {start_epoch+1}–{epochs}')
    for ep in range(start_epoch, epochs):
        model.train()
        ep_loss = 0.0
        pbar = tqdm(train_loader, desc=f'{name} ep{ep+1}/{epochs}', leave=False)
        for batch in pbar:
            ids = batch.to(DEVICE)
            opt.zero_grad()
            loss = model(ids, labels=ids).loss
            loss.backward()
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
            opt.step(); sched.step()
            ep_loss += loss.item(); step += 1
            pbar.set_postfix(loss=f'{loss.item():.4f}')
            if step % EVAL_EVERY == 0:
                vp = evaluate_ppl(model, val_loader)
                log['step'].append(step); log['train_loss'].append(loss.item()); log['val_ppl'].append(vp)
                model.train()

        avg = ep_loss / len(train_loader)
        vp  = evaluate_ppl(model, val_loader)
        print(f'  ep{ep+1}  avg_loss={avg:.4f}  val_ppl={vp:.2f}')
        sname   = _safe_name(name)
        ep_ckpt = CKPT_DIR / f'{sname}_epoch{ep+1}.pth'
        torch.save({'epoch': ep+1, 'state': model.state_dict(),
                    'optimizer': opt.state_dict(), 'scheduler': sched.state_dict(), 'log': log}, ep_ckpt)
        prev = CKPT_DIR / f'{sname}_epoch{ep}.pth'
        if prev.exists(): prev.unlink()

    if ckpt_path:
        torch.save({'state': model.state_dict(), 'log': log, 'epochs_done': epochs}, ckpt_path)
        print(f'  Saved → {ckpt_path}')
    return log

print('Training loop ready.')

## 6. Load v2 Results + Train PNG
**v2 variants (Baseline, G1–G5)** — loaded from checkpoint, no training.
**PNG** — trained fresh in this notebook.

In [ ]:
# All 7 variants: 6 from v2 (load only) + 1 new (PNG, train here)
EXPERIMENTS = [
    ('Baseline',          'baseline'),
    ('G1 — SDPA output',  'G1'),
    ('G2 — Value proj',   'G2'),
    ('G3 — Key proj',     'G3'),
    ('G4 — Query proj',   'G4'),
    ('G5 — Dense output', 'G5'),
    ('PNG — Norm-Aware',  'PNG'),
]

# Load v2 results JSON if available (has test PPL for G1–G5 + Baseline)
_v2_results_file = V2_CKPT_DIR / 'results.json'
_v3_results_file = CKPT_DIR / 'results.json'

v2_results_saved = {}
if _v2_results_file.exists():
    with open(_v2_results_file) as f:
        v2_results_saved = json.load(f)
    print(f'Loaded v2 results: {list(v2_results_saved.keys())}')

if _v3_results_file.exists():
    with open(_v3_results_file) as f:
        v3_results_saved = json.load(f)
    print(f'Loaded v3 results: {list(v3_results_saved.keys())}')
else:
    v3_results_saved = {}

# Merge: v2 results are the source of truth for G1–G5 + Baseline
results_saved = {**v2_results_saved, **v3_results_saved}

results = {}

for name, pos in EXPERIMENTS:
    sname = _safe_name(name)
    ckpt  = CKPT_DIR / f'gpt2_{sname}.pth'
    print(f'\n── {name} ──')

    # ── Load-only for v2 variants ─────────────────────────────────────────
    if pos != 'PNG':
        if ckpt.exists() and name in results_saved:
            data = torch.load(ckpt, map_location='cpu', weights_only=False)
            results[name] = {'log': data['log'], 'test_ppl': results_saved[name]['test_ppl']}
            print(f'  [LOADED] test_ppl={results_saved[name]["test_ppl"]:.2f}')
        else:
            print(f'  WARNING: checkpoint or results missing for {name}')
            print(f'    ckpt exists: {ckpt.exists()}  |  in results_saved: {name in results_saved}')
            # Use placeholder so figures still render
            results[name] = {'log': {'step':[],'train_loss':[],'val_ppl':[]}, 'test_ppl': float('nan')}
        continue

    # ── Train PNG ─────────────────────────────────────────────────────────
    if ckpt.exists():
        data = torch.load(ckpt, map_location='cpu', weights_only=False)
        epochs_done = data.get('epochs_done', 0)
        if epochs_done >= EPOCHS and name in results_saved:
            print(f'  [SKIP] {epochs_done} epochs done')
            results[name] = {'log': data['log'], 'test_ppl': results_saved[name]['test_ppl']}
            continue

    model = build_model('PNG')
    epoch_ckpts = sorted(_glob.glob(str(CKPT_DIR / f'{sname}_epoch*.pth')))
    resume = epoch_ckpts[-1] if epoch_ckpts else None
    log = train_one(model, name=name, ckpt_path=ckpt, resume_ckpt=resume)

    test_ppl = evaluate_ppl(model, test_loader)
    print(f'  test PPL = {test_ppl:.2f}')
    results[name] = {'log': log, 'test_ppl': test_ppl}
    results_saved[name] = {'test_ppl': test_ppl}
    with open(_v3_results_file, 'w') as f:
        json.dump({k: v for k, v in results_saved.items() if k == 'PNG — Norm-Aware'}, f, indent=2)

    del model
    if DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'\nAll {len(EXPERIMENTS)} variants ready.')

## 7. Analysis Helpers

In [ ]:
def load_for_analysis(name, pos):
    sname = _safe_name(name)
    ckpt  = CKPT_DIR / f'gpt2_{sname}.pth'
    if not ckpt.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt}')
    m = build_model(pos)
    data = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    m.load_state_dict(data['state'])
    m.eval()
    return m

print('load_for_analysis() ready.')

## 8. Attention Sink Analysis

In [ ]:
def get_first_token_attn(model, loader, n_batches=SINK_BATCHES):
    n_layers = model.config.n_layer
    n_heads  = model.config.n_head
    head_dim = model.config.n_embd // n_heads
    all_first = [[] for _ in range(n_layers)]
    handles   = []
    for li, block in enumerate(model.transformer.h):
        cap = {}
        def make_hook(idx, c):
            def hook(module, inp, out):
                hs = inp[0]; B, N, C = hs.shape
                qkv = module.c_attn(hs)
                q, k, _ = qkv.split(module.embed_dim, dim=2)
                q = _split_heads(q, n_heads, head_dim)
                k = _split_heads(k, n_heads, head_dim)
                scores = (q * (head_dim ** -0.5)) @ k.transpose(-2, -1)
                cm = torch.triu(torch.ones(N, N, device=hs.device, dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None, None], float('-inf'))
                w = scores.softmax(-1)
                c['first'] = w[:, :, :, 0].mean(dim=(1, 2)).detach()
            return hook
        h = block.attn.register_forward_hook(make_hook(li, cap))
        handles.append((li, cap, h))
    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            model(batch.to(DEVICE))
            for li, cap, _ in handles:
                if 'first' in cap:
                    all_first[li].append(cap['first'].mean().item())
    for _, _, h in handles: h.remove()
    return [np.mean(v) if v else 0.0 for v in all_first]


sink = {}
for _name, _pos in [('Baseline', 'baseline'), ('G1 — SDPA output', 'G1'), ('PNG — Norm-Aware', 'PNG')]:
    print(f'Attention sink: {_name}...')
    _m = load_for_analysis(_name, _pos)
    sink[_name] = get_first_token_attn(_m, val_loader)
    del _m
    if DEVICE == 'cuda': torch.cuda.empty_cache()

for _name, _vals in sink.items():
    print(f'  {_name:<26} mean={np.mean(_vals):.4f}')

## 9. Gate Sparsity & Gate-vs-Norm Analysis

In [ ]:
@torch.no_grad()
def collect_gate_scores(model, loader, n_batches=GATE_BATCHES):
    for block in model.transformer.h:
        block.attn._capture_gate = True
    all_gates = [[] for _ in range(model.config.n_layer)]
    model.eval()
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        model(batch.to(DEVICE))
        for li, block in enumerate(model.transformer.h):
            if hasattr(block.attn, '_last_gate'):
                all_gates[li].append(block.attn._last_gate.mean().item())
    for block in model.transformer.h:
        block.attn._capture_gate = False
    return [np.mean(v) if v else 0.0 for v in all_gates]


@torch.no_grad()
def collect_gate_vs_norm(model, loader, n_batches=GATE_BATCHES):
    last = model.transformer.h[-1]
    last.attn._capture_gate = True
    all_norms, all_gates = [], []
    model.eval()
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        model(batch.to(DEVICE))
        if hasattr(last.attn, '_last_gate'):
            all_gates.append(last.attn._last_gate.mean(-1).cpu().flatten().numpy())
            all_norms.append(last.attn._last_x_norm.cpu().flatten().numpy())
    last.attn._capture_gate = False
    if not all_gates: return np.array([]), np.array([])
    return np.concatenate(all_norms), np.concatenate(all_gates)


gate_scores = {}
gate_norm   = {}

for _name, _pos in [('G1 — SDPA output', 'G1'), ('PNG — Norm-Aware', 'PNG')]:
    print(f'Gate analysis: {_name}...')
    _m = load_for_analysis(_name, _pos)
    gate_scores[_name] = collect_gate_scores(_m, val_loader)
    _norms, _gates      = collect_gate_vs_norm(_m, val_loader)
    gate_norm[_name]    = (_norms, _gates)
    del _m
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    print(f'  per-layer: {[f"{s:.3f}" for s in gate_scores[_name]]}')

# Pearson r for G1 and PNG
pearson = {}
for _name, (_norms, _gates) in gate_norm.items():
    if len(_norms) > 1:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(_norms), min(5000, len(_norms)), replace=False)
        r, p = pearsonr(_norms[idx], _gates[idx])
        pearson[_name] = (r, p, _norms[idx], _gates[idx])
        print(f'  {_name}  r={r:.3f}  p={p:.2e}')
    else:
        pearson[_name] = (float('nan'), float('nan'), np.array([]), np.array([]))

## 10. Word-Level Attention Matrices

In [ ]:
for ci in range(len(val_ids)):
    text = tokenizer.decode(val_ids[ci].tolist(), skip_special_tokens=True)
    if len(text.strip()) > 80 and '=' not in text[:30]:
        SAMPLE_CHUNK_IDX = ci
        break

SAMPLE_IDS = val_ids[SAMPLE_CHUNK_IDX].unsqueeze(0)
sample_tokens  = tokenizer.convert_ids_to_tokens(SAMPLE_IDS[0, :DISPLAY_LEN].tolist())
display_tokens = [t.replace('\u0120', ' ').replace('\u010a', '\\n') for t in sample_tokens]
print(f'Sample [{SAMPLE_CHUNK_IDX}]: {" ".join(display_tokens)}')


def _get_attn_via_hook(model, input_ids, n_tokens=DISPLAY_LEN):
    """Hook-based fallback: manually computes softmax attention weights."""
    n_heads  = model.config.n_head
    head_dim = model.config.n_embd // n_heads
    attn_layers, handles = [], []
    for block in model.transformer.h:
        cap = {}
        def make_hook(c):
            def hook(module, inp, out):
                hs = inp[0]; B, N, _ = hs.shape
                qkv = module.c_attn(hs)
                q, k, _ = qkv.split(module.embed_dim, dim=2)
                q = q.view(B, N, n_heads, head_dim).permute(0,2,1,3)
                k = k.view(B, N, n_heads, head_dim).permute(0,2,1,3)
                scores = (q * head_dim**-0.5) @ k.transpose(-2,-1)
                cm = torch.triu(torch.ones(N, N, device=hs.device, dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(cm[None,None], float('-inf'))
                c['attn'] = scores.softmax(-1)[0].detach().cpu().numpy()
            return hook
        h = block.attn.register_forward_hook(make_hook(cap))
        handles.append((cap, h))
    model.eval()
    with torch.no_grad():
        model(input_ids.to(DEVICE))
    for cap, h in handles:
        h.remove()
        if 'attn' in cap:
            attn_layers.append(cap['attn'])
    if not attn_layers:
        return np.zeros((n_tokens, n_tokens))
    return np.stack(attn_layers).mean(axis=(0,1))[:n_tokens, :n_tokens]


def get_attn_matrix(model, input_ids, n_tokens=DISPLAY_LEN, is_baseline=False):
    model.eval()
    if is_baseline:
        # Disable flash attn so output_attentions=True actually returns weights
        _orig_impl = getattr(model.config, '_attn_implementation', 'sdpa')
        model.config._attn_implementation = 'eager'
        with torch.no_grad():
            out = model(input_ids.to(DEVICE), output_attentions=True)
        model.config._attn_implementation = _orig_impl
        if out.attentions is None:
            print('  output_attentions still None — using hook fallback')
            return _get_attn_via_hook(model, input_ids, n_tokens)
        attn_layers = [a[0].cpu().numpy() for a in out.attentions]
    else:
        for block in model.transformer.h:
            block.attn._capture_attn = True
        with torch.no_grad():
            model(input_ids.to(DEVICE))
        attn_layers = []
        for block in model.transformer.h:
            if hasattr(block.attn, '_last_attn'):
                attn_layers.append(block.attn._last_attn[0].cpu().numpy())
            block.attn._capture_attn = False
    if not attn_layers:
        print('  WARNING: no attention captured')
        return np.zeros((n_tokens, n_tokens))
    return np.stack(attn_layers).mean(axis=(0,1))[:n_tokens, :n_tokens]


attn_mats = {}
for _name, _pos, _is_base in [
    ('Baseline',         'baseline', True),
    ('G1 — SDPA output', 'G1',       False),
    ('PNG — Norm-Aware', 'PNG',       False),
]:
    print(f'Attn matrix: {_name}...')
    _m = load_for_analysis(_name, _pos)
    attn_mats[_name] = get_attn_matrix(_m, SAMPLE_IDS, is_baseline=_is_base)
    del _m
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    print(f'  max={attn_mats[_name].max():.4f}  min={attn_mats[_name].min():.4f}')

## 11. Figures

In [ ]:
sns.set_theme(style='whitegrid', font_scale=1.05)
names        = [n for n, _ in EXPERIMENTS]
ppls         = [results[n]['test_ppl'] for n in names]
baseline_ppl = results['Baseline']['test_ppl']
deltas       = [baseline_ppl - p for p in ppls]

palette_var = {
    'Baseline':          '#888888',
    'G1 — SDPA output':  '#4C72B0',
    'G2 — Value proj':   '#55A868',
    'G3 — Key proj':     '#C44E52',
    'G4 — Query proj':   '#DD8452',
    'G5 — Dense output': '#937860',
    'PNG — Norm-Aware':  '#9467BD',
}

In [ ]:
# ── Fig A — PPL ablation bar chart (all 7 variants) ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Fig A  |  Gate Position Ablation — Test PPL\n'
             '(GPT-2 small, WikiText-103, unfrozen backbone, v3 + PNG)',
             fontsize=13, fontweight='bold')

colors = [palette_var[n] for n in names]
ax = axes[0]
bars = ax.bar(names, ppls, color=colors, edgecolor='k', linewidth=0.6)
for bar, v in zip(bars, ppls):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{v:.2f}', ha='center', va='bottom', fontsize=7)
ax.axhline(baseline_ppl, color='gray', linestyle='--', lw=1.2, label='Baseline')
ax.set_ylabel('Test Perplexity (↓ better)')
ax.set_title('Absolute Test PPL', fontweight='bold')
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylim(min(p for p in ppls if not np.isnan(p))*0.97,
            max(p for p in ppls if not np.isnan(p))*1.01)
ax.grid(axis='y', alpha=0.3); ax.legend()

ax2 = axes[1]
dcol = ['#2ca02c' if d > 0 else ('#d62728' if d < 0 else '#888888') for d in deltas]
bars2 = ax2.bar(names, deltas, color=dcol, edgecolor='k', linewidth=0.6)
for bar, d in zip(bars2, deltas):
    if not np.isnan(d):
        ax2.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height() + (0.05 if d >= 0 else -0.25),
                 f'{d:+.2f}', ha='center', va='bottom', fontsize=7)
ax2.axhline(0, color='k', lw=1)
ax2.set_ylabel('ΔPPL vs Baseline (↑ = better)')
ax2.set_title('PPL Improvement over Baseline', fontweight='bold')
ax2.set_xticklabels(names, rotation=30, ha='right')
ax2.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig('figA_ppl_ablation_v3.pdf', bbox_inches='tight')
plt.show()
print('Saved figA_ppl_ablation_v3.pdf')

In [ ]:
# ── Fig B — Training curves (PNG only — others loaded from v2) ──────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig B  |  Training Curves — PNG gate\n'
             '(G1–G5 curves from v2 shown for reference; PNG trained in v3)',
             fontsize=13, fontweight='bold')

for name, _ in EXPERIMENTS:
    log = results[name]['log']
    if log['step']:
        lw  = 2.5 if name == 'PNG — Norm-Aware' else 1.0
        ls  = '-'  if name == 'PNG — Norm-Aware' else '--'
        ax1.plot(log['step'], log['train_loss'], color=palette_var[name],
                 label=name, alpha=0.9, lw=lw, ls=ls)
        ax2.plot(log['step'], log['val_ppl'], color=palette_var[name],
                 label=name, alpha=0.9, lw=lw, ls=ls)

ax1.set_xlabel('Step'); ax1.set_ylabel('Train Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.legend(fontsize=7); ax1.grid(alpha=0.3)

ax2.set_xlabel('Step'); ax2.set_ylabel('Val PPL')
ax2.set_title('Validation Perplexity', fontweight='bold')
ax2.legend(fontsize=7); ax2.grid(alpha=0.3)

fig.tight_layout()
fig.savefig('figB_training_curves_v3.pdf', bbox_inches='tight')
plt.show()
print('Saved figB_training_curves_v3.pdf')

In [ ]:
# ── Fig C — Attention Sink (Baseline vs G1 vs PNG) ─────────────────────────
layers_x = list(range(1, len(list(sink.values())[0]) + 1))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig C  |  Attention Sink — First-Token Attention Score\n'
             'Baseline vs G1 vs PNG  (GPT-2 small, WikiText-103 val)',
             fontsize=13, fontweight='bold')

markers = {'Baseline': 'o', 'G1 — SDPA output': 's', 'PNG — Norm-Aware': '^'}
for _name, _vals in sink.items():
    ax1.plot(layers_x, _vals, marker=markers.get(_name,'o'), linestyle='-',
             color=palette_var[_name], label=f'{_name} (mean={np.mean(_vals):.3f})')
    ax1.axhline(np.mean(_vals), color=palette_var[_name], linestyle='--', lw=0.8)

ax1.set_xlabel('Layer'); ax1.set_ylabel('Mean Attention to First Token')
ax1.set_title('Per-Layer Attention Sink', fontweight='bold')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2.bar(list(sink.keys()), [np.mean(v) for v in sink.values()],
        color=[palette_var[n] for n in sink.keys()], edgecolor='k')
for i, (n, v) in enumerate(sink.items()):
    ax2.text(i, np.mean(v)+0.001, f'{np.mean(v):.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax2.set_ylabel('Mean First-Token Attention')
ax2.set_title('Attention Sink Comparison', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig('figC_attention_sink_v3.pdf', bbox_inches='tight')
plt.show()
print('Saved figC_attention_sink_v3.pdf')

In [ ]:
# ── Fig D — Gate Sparsity per Layer: G1 vs PNG ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
fig.suptitle('Fig D  |  Gate Sparsity per Layer — G1 vs PNG\n'
             'Mean gate score < 0.5 → active suppression of SDPA outputs',
             fontsize=13, fontweight='bold')

for ax, (_name, _color) in zip(axes, [('G1 — SDPA output','#4C72B0'), ('PNG — Norm-Aware','#9467BD')]):
    scores = gate_scores.get(_name, [])
    ax.bar(range(1, len(scores)+1), scores, color=_color, edgecolor='k', linewidth=0.5)
    ax.axhline(0.5, color='red', linestyle='--', lw=1.2, label='σ=0.5 (neutral)')
    ax.set_xlabel('Layer'); ax.set_ylabel('Mean Gate Score')
    ax.set_title(_name, fontweight='bold')
    ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig('figD_gate_sparsity_v3.pdf', bbox_inches='tight')
plt.show()
print('Saved figD_gate_sparsity_v3.pdf')

In [ ]:
# ── Fig E — Gate vs Norm scatter: G1 vs PNG ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fig E  |  Gate Score vs Token Norm — G1 vs PNG (last layer)\n'
             'PNG explicitly conditions on norm; expect stronger correlation',
             fontsize=13, fontweight='bold')

for ax, _name in zip(axes, ['G1 — SDPA output', 'PNG — Norm-Aware']):
    r, p, ns, gs = pearson.get(_name, (float('nan'), float('nan'), np.array([]), np.array([])))
    if len(ns) > 0:
        ax.scatter(ns, gs, alpha=0.15, s=6, color=palette_var[_name], rasterized=True)
        z  = np.polyfit(ns, gs, 1)
        xr = np.linspace(ns.min(), ns.max(), 200)
        ax.plot(xr, np.polyval(z, xr), 'r-', lw=2, label=f'r={r:.3f}  p={p:.1e}')
        ax.legend(fontsize=10)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
    ax.set_xlabel('Token L2 Norm  ||x||₂', fontsize=11)
    ax.set_ylabel('Gate Score  σ(·)', fontsize=11)
    ax.set_title(_name, fontweight='bold')
    ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig('figE_gate_vs_norm_v3.pdf', bbox_inches='tight')
plt.show()
print('Saved figE_gate_vs_norm_v3.pdf')

In [ ]:
# ── Fig F — Layer × Head Heatmap: G1 vs PNG ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 5))
fig.suptitle('Fig F  |  Gate Activations — Layer × Head  (G1 vs PNG)\n'
             'GPT-2 small, WikiText-103 val',
             fontsize=13, fontweight='bold')

for ax, (_name, _pos) in zip(axes, [('G1 — SDPA output','G1'), ('PNG — Norm-Aware','PNG')]):
    print(f'Gate heatmap: {_name}...')
    _m = load_for_analysis(_name, _pos)
    for block in _m.transformer.h:
        block.attn._capture_gate = True
    n_layers = _m.config.n_layer; n_heads = _m.config.n_head
    lh = np.zeros((n_layers, n_heads)); nc = 0
    _m.eval()
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= GATE_BATCHES: break
            _m(batch.to(DEVICE))
            for li, block in enumerate(_m.transformer.h):
                if hasattr(block.attn, '_last_gate'):
                    lh[li] += block.attn._last_gate.mean(dim=(0,1)).cpu().numpy()
            nc += 1
    lh /= max(nc, 1)
    del _m
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    sns.heatmap(lh, annot=True, fmt='.2f', cmap='RdYlGn',
                xticklabels=[f'H{i+1}' for i in range(n_heads)],
                yticklabels=[f'L{i+1}' for i in range(n_layers)],
                vmin=0, vmax=1, ax=ax, linewidths=0.4)
    ax.set_xlabel('Head'); ax.set_ylabel('Layer')
    ax.set_title(_name, fontweight='bold')

fig.tight_layout()
fig.savefig('figF_gate_heatmap_v3.pdf', bbox_inches='tight')
plt.show()
print('Saved figF_gate_heatmap_v3.pdf')

In [ ]:
# ── Fig G — Word-Level Attention Heatmap (Baseline / G1 / PNG) ──────────────
VARIANTS_ATTN = [
    ('Baseline',         attn_mats['Baseline']),
    ('G1 — SDPA output', attn_mats['G1 — SDPA output']),
    ('PNG — Norm-Aware', attn_mats['PNG — Norm-Aware']),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle(
    'Fig G  |  Word-Level Attention Heatmap\n'
    'Mean over all layers & heads · GPT-2 small · WikiText-103 val',
    fontsize=13, fontweight='bold'
)
for ax, (vname, mat) in zip(axes, VARIANTS_ATTN):
    mask = np.triu(np.ones_like(mat, dtype=bool), k=1)
    sns.heatmap(mat, ax=ax, mask=mask, cmap='YlOrRd',
                xticklabels=display_tokens, yticklabels=display_tokens,
                cbar=True, linewidths=0.0, vmin=0)
    ax.set_title(vname, fontsize=12, fontweight='bold')
    ax.set_xlabel('Key token (attended to)', fontsize=9)
    ax.set_ylabel('Query token (attending from)', fontsize=9)
    ax.set_xticklabels(display_tokens, rotation=90, fontsize=6)
    ax.set_yticklabels(display_tokens, rotation=0, fontsize=6)

fig.tight_layout()
fig.savefig('figG_word_attention_heatmap_v3.pdf', bbox_inches='tight')
plt.show()
print('Saved figG_word_attention_heatmap_v3.pdf')

In [ ]:
# ── Fig H — Per-Token Attention Received ────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(18, 9))
fig.suptitle(
    'Fig H  |  Per-Token Attention Received (Column Sum)\n'
    'How much each word is attended to — averaged over all query positions & heads & layers',
    fontsize=13, fontweight='bold'
)
for ax, (vname, mat) in zip(axes, VARIANTS_ATTN):
    col_sum = mat.sum(axis=0); col_sum = col_sum / col_sum.sum()
    bar_colors = plt.cm.YlOrRd(col_sum / col_sum.max())
    ax.bar(range(len(display_tokens)), col_sum, color=bar_colors, edgecolor='none')
    ax.set_xticks(range(len(display_tokens)))
    ax.set_xticklabels(display_tokens, rotation=70, ha='right', fontsize=7)
    ax.set_ylabel('Attention weight')
    ax.set_title(vname, fontweight='bold', fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    top3 = np.argsort(col_sum)[-3:][::-1]
    for t in top3:
        ax.text(t, col_sum[t]+0.002, '▲', ha='center', va='bottom', fontsize=8, color='red')

fig.tight_layout()
fig.savefig('figH_per_token_attn_v3.pdf', bbox_inches='tight')
plt.show()
print('Saved figH_per_token_attn_v3.pdf')

## 12. Results Summary

In [ ]:
print('=' * 75)
print('  RESULTS SUMMARY  (v3 — G1–G5 from v2 + PNG trained here)')
print(f'  Model  : GPT-2 small (124M) — UNFROZEN backbone')
print(f'  Data   : WikiText-103  |  Epochs: {EPOCHS}  |  LR: {LR}')
print('=' * 75)
print(f'  {"Method":<26} {"Test PPL":>10} {"ΔPPL":>8}')
print('  ' + '-' * 50)
best_ppl = min(p for p in ppls if not np.isnan(p))
for name, ppl, delta in zip(names, ppls, deltas):
    tag = '  ← best' if ppl == best_ppl else ''
    tag += '  ← PNG' if name == 'PNG — Norm-Aware' else ''
    ppl_s   = f'{ppl:.2f}'   if not np.isnan(ppl)   else 'N/A'
    delta_s = f'{delta:+.2f}' if not np.isnan(delta) else 'N/A'
    print(f'  {name:<26} {ppl_s:>10} {delta_s:>8}{tag}')

print()
print('  Attention Sink (mean first-token attn):')
for _n, _v in sink.items():
    print(f'    {_n:<26}  {np.mean(_v):.4f}')

print()
print('  Gate-Norm Pearson r (last layer):')
for _n, (r, p, _, _) in pearson.items():
    print(f'    {_n:<26}  r={r:.3f}  p={p:.2e}')

print()
print('  Figures saved (v3):')
for fname in ['figA_ppl_ablation_v3.pdf', 'figB_training_curves_v3.pdf',
              'figC_attention_sink_v3.pdf', 'figD_gate_sparsity_v3.pdf',
              'figE_gate_vs_norm_v3.pdf',  'figF_gate_heatmap_v3.pdf',
              'figG_word_attention_heatmap_v3.pdf', 'figH_per_token_attn_v3.pdf']:
    ok = '✓' if Path(fname).exists() else '✗'
    print(f'    {ok} {fname}')
print('=' * 75)